In [ ]:
from sqlalchemy import create_engine
import pandas as pd

# Reading MAXQDA export

In [ ]:
mqda_file = "../data/maxqda/prin_22_stern_1_2.mqda"

engine = create_engine("sqlite:///" + mqda_file)

In [ ]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", engine)

# Label extraction

These are uncategorised labels, ordered by length.

In [ ]:
df_codes = pd.read_sql("SELECT * FROM Codewords", engine)
codenames = sorted(set(df_codes["Name"]), key=lambda x: -len(x))
codenames

# Coding hierarchy extraction

Extract the tree structure of the coding system using `WordID` as parent pointer (`WordID=0` = root).

In [ ]:
import sqlite3

conn = sqlite3.connect(mqda_file)
cur = conn.cursor()

# Deduplicate codes (some IDs appear multiple times with different Idx values)
cur.execute("""
    SELECT ID, Name, WordID, MIN(Idx) as SortIdx
    FROM Codewords
    GROUP BY ID, Name, WordID
    ORDER BY WordID, SortIdx
""")
rows = cur.fetchall()

# Build tree: WordID is the parent pointer (WordID=0 = root)
nodes: dict[int, dict] = {}
children_of: dict[int, list[int]] = {}
for row_id, name, parent_id, sort_idx in rows:
    nodes[row_id] = {"name": name.strip(), "sort_idx": sort_idx}
    children_of.setdefault(parent_id, []).append(row_id)

# Sort children by their Idx position
for parent_id in children_of:
    children_of[parent_id].sort(key=lambda cid: nodes[cid]["sort_idx"])

conn.close()


def print_tree(parent_id: int = 0, indent: int = 0) -> None:
    for child_id in children_of.get(parent_id, []):
        print("  " * indent + nodes[child_id]["name"])
        print_tree(child_id, indent + 1)


print(f"{len(nodes)} unique codes, {len(children_of.get(0, []))} root categories\n")
print_tree()

# Export as Mermaid mindmap

Generate `docs/mqda_coding_taxonomy.mmd` — a Mermaid mindmap of the full coding taxonomy.
Labels containing parentheses, quotes, or brackets are wrapped in backticks to avoid Mermaid shape parsing.

In [ ]:
lines = ["mindmap", "  root((MQDA Coding System))"]


def add_mermaid_nodes(parent_id: int, depth: int) -> None:
    for child_id in children_of.get(parent_id, []):
        name = nodes[child_id]["name"]
        indent = "  " * (depth + 2)
        name = name.replace("`", "'")
        lines.append(f"{indent}{name}")
        add_mermaid_nodes(child_id, depth + 1)


add_mermaid_nodes(0, 0)

mmd_path = "../docs/mqda_coding_taxonomy.mmd"
with open(mmd_path, "w") as f:
    f.write("\n".join(lines) + "\n")

print(f"Written {len(lines)} lines to {mmd_path}")